# Create Augmented Dataset

Generates **20 augmented Dixon MRI triplets** (water + fat + segmentation mask)
from the myosegmenTUM dataset using `dissector.creation.generate_augmented`.

- **10 samples** drawn from `P*` subjects (patients)
- **10 samples** drawn from `HV*` subjects (healthy volunteers)

Each source scan is augmented once (`n=1` per call).  Spatial transforms
(flip, affine, elastic deformation) are applied **jointly** to the image and
mask so they stay aligned; intensity transforms (bias field, noise, gamma)
are applied to images only.

Output lands in `eval_notebooks/our_augmented_dataset/`.

In [ ]:
import SimpleITK as sitk
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

In [ ]:
import logging
import pathlib
import random

from dissector.creation import generate_augmented

logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')

In [ ]:
MYOSEG_DIR = pathlib.Path(r'C:\Projects\dissector\eval_notebooks\myosegmenTUM')
OUTPUT_DIR = pathlib.Path(r'C:\Projects\dissector\eval_notebooks\our_augmented_dataset')

N_PER_GROUP = 10   # 10 P* + 10 HV* = 20 total
SEED        = 42


def find_stacks(subject_dir: pathlib.Path) -> list[tuple[pathlib.Path, pathlib.Path, pathlib.Path | None, str]]:
    """Return [(water_path, fat_path, seg_path, stem), ...] for every stack in subject_dir."""
    name      = subject_dir.name
    water_dir = subject_dir / 'ImageData' / f'{name}_WATER'
    fat_dir   = subject_dir / 'ImageData' / f'{name}_FATFRACTION'
    seg_dir   = subject_dir / 'SegmentationMasks'
    stacks = []
    if not water_dir.is_dir() or not fat_dir.is_dir():
        return stacks
    for water_file in sorted(water_dir.glob(f'{name}_WATER_stack*.nii')):
        stack_num = water_file.stem.split('_stack')[-1]
        fat_file  = fat_dir / f'{name}_FATFRACTION_stack{stack_num}.nii'
        seg_file  = seg_dir / f'combined_gt_stack{stack_num}.mha'
        if fat_file.exists():
            stacks.append((
                water_file,
                fat_file,
                seg_file if seg_file.exists() else None,
                f'{name}_stack{stack_num}',
            ))
    return stacks


p_stacks, hv_stacks = [], []
for subj in sorted(MYOSEG_DIR.iterdir()):
    if not subj.is_dir():
        continue
    stacks = find_stacks(subj)
    if subj.name.startswith('P'):
        p_stacks.extend(stacks)
    elif subj.name.startswith('HV'):
        hv_stacks.extend(stacks)

print(f'P*  stacks found : {len(p_stacks)}')
for w, f, s, stem in p_stacks:
    print(f'  {stem}  seg={"✓" if s else "✗"}')
print(f'\nHV* stacks found : {len(hv_stacks)}')
for w, f, s, stem in hv_stacks:
    print(f'  {stem}  seg={"✓" if s else "✗"}')

In [ ]:
rng = random.Random(SEED)

selected_p  = rng.sample(p_stacks,  min(N_PER_GROUP, len(p_stacks)))
selected_hv = rng.sample(hv_stacks, min(N_PER_GROUP, len(hv_stacks)))

print(f'Selected {len(selected_p)} P*  sources and {len(selected_hv)} HV* sources')
print('\nP* selection:')
for _, _, seg, s in selected_p:
    print(f'  {s}  seg={"✓" if seg else "✗"}')
print('\nHV* selection:')
for _, _, seg, s in selected_hv:
    print(f'  {s}  seg={"✓" if seg else "✗"}')

In [ ]:
all_sources = selected_p + selected_hv
all_outputs = []   # list of (water_out, fat_out, seg_out | None)

for i, (water_path, fat_path, seg_path, stem) in enumerate(all_sources, 1):
    group = 'P*' if stem.startswith('P') else 'HV*'
    print(f'[{i:02d}/{len(all_sources)}] {group}  {stem}  seg={"✓" if seg_path else "✗"}')
    outputs = generate_augmented(
        water_path=water_path,
        fat_path=fat_path,
        n=1,
        output_dir=OUTPUT_DIR,
        stem=stem,
        seg_path=seg_path,
    )
    all_outputs.extend(outputs)

print(f'\nDone.  {len(all_outputs)} augmented triplets saved to:')
print(f'  {OUTPUT_DIR}')

In [ ]:
print('Output files:')
for water_out, fat_out, seg_out in all_outputs:
    print(f'  water : {water_out.name}')
    print(f'  fat   : {fat_out.name}')
    print(f'  seg   : {seg_out.name if seg_out else "—"}')
    print()

In [ ]:


# myosegmenTUM combined_gt label convention (from creation.py docstring)
GT_LABEL_COLOURS = {
    1: ('L_Gracilis',   (0.12, 0.47, 0.71)),
    2: ('L_Hamstrings', (0.20, 0.63, 0.17)),
    3: ('L_Quadriceps', (0.89, 0.10, 0.11)),
    4: ('L_Sartorius',  (1.00, 0.50, 0.00)),
    5: ('R_Gracilis',   (0.60, 0.39, 0.64)),
    6: ('R_Hamstrings', (0.55, 0.34, 0.29)),
    7: ('R_Quadriceps', (0.89, 0.47, 0.76)),
    8: ('R_Sartorius',  (0.74, 0.74, 0.13)),
}

SEG_ALPHA = 0.45
NCOLS     = 4
NROWS     = (len(all_outputs) + NCOLS - 1) // NCOLS

fig, axes = plt.subplots(NROWS, NCOLS, figsize=(NCOLS * 4, NROWS * 4))

for ax, (water_out, fat_out, seg_out) in zip(axes.flat, all_outputs):
    # Load water image
    water_arr = sitk.GetArrayFromImage(sitk.ReadImage(str(water_out)))
    mid       = water_arr.shape[0] // 2
    img       = water_arr[mid].astype(np.float32)
    lo, hi    = np.percentile(img, 1), np.percentile(img, 99)
    img_norm  = np.clip((img - lo) / (hi - lo + 1e-8), 0, 1)

    ax.imshow(img_norm, cmap='gray', origin='lower')

    # Overlay segmentation if available
    if seg_out is not None and seg_out.exists():
        seg_arr   = sitk.GetArrayFromImage(sitk.ReadImage(str(seg_out)))
        seg_slice = seg_arr[mid]
        rgba      = np.zeros((*seg_slice.shape, 4), dtype=np.float32)
        for label, (_, colour) in GT_LABEL_COLOURS.items():
            mask = seg_slice == label
            if mask.any():
                rgba[mask] = (*colour, SEG_ALPHA)
        ax.imshow(rgba, origin='lower')

    title = water_out.name.replace('_augmented000_water.nii.gz', '')
    ax.set_title(title, fontsize=7)
    ax.axis('off')

# Hide unused axes
for ax in axes.flat[len(all_outputs):]:
    ax.axis('off')

# Shared legend
patches = [
    mpatches.Patch(color=colour, alpha=0.8, label=name)
    for _, (name, colour) in GT_LABEL_COLOURS.items()
]
fig.legend(handles=patches, loc='lower center', fontsize=8, ncol=4,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Augmented dataset — water image + GT overlay (middle slice)', fontsize=13)
plt.tight_layout()
plt.show()